In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets trl

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
from datasets import load_dataset
dataset = load_dataset("philschmid/gretel-synthetic-text-to-sql")

In [ ]:
dataset["train"] = dataset["train"].shuffle(seed=42).select(range(4000))
dataset["test"] = dataset["test"].shuffle(seed=42).select(range(300))

from transformers import AutoTokenizer
model_id = "Qwen/Qwen2-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def format_example(example):
    system_msg = "You are a helpful assistant that writes SQL queries given a database schema and a question."
    user_msg = f"### Database Schema:\n{example['sql_context']}\n\n### Question:\n{example['sql_prompt']}"
    assistant_msg = example["sql"]
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_msg},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

dataset = dataset.map(format_example)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

trainable params: 10,092,544 || all params: 7,625,709,056 || trainable%: 0.1323


In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.float16
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

## Training (already completed — adapter uploaded to Hugging Face)
Cells below show the training run that produced the adapter. Not re-executed in this notebook; for inference/eval only, adapter is loaded directly from the Hub in the section further down.

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="/content/drive/MyDrive/qwen2-7b-sql-qlora-checkpoints",  # saves to Drive now
    num_train_epochs=2,   # back to 2, saves ~50 min vs 3
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    fp16=False,
    bf16=False,
    max_length=512,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
)

trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

...

Building labels for eval dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,1.395201
20,0.678454
30,0.509351
40,0.488460
50,0.483836
60,0.489013
70,0.459722
80,0.471310
90,0.459455
100,0.464001


TrainOutput(global_step=500, training_loss=0.4658120174407959, metrics={'train_runtime': 8203.6663, 'train_samples_per_second': 0.975, 'train_steps_per_second': 0.061, 'total_flos': 7.273213572301824e+16, 'train_loss': 0.4658120174407959, 'epoch': 2.0})

In [ ]:
trainer.save_model("/content/drive/MyDrive/qwen2-7b-sql-qlora-final")
tokenizer.save_pretrained("/content/drive/MyDrive/qwen2-7b-sql-qlora-final")
print("Saved to Google Drive — safe this time!")

Saved to Google Drive — safe this time!


In [ ]:
import sqlite3

def create_db_from_schema(schema_sql):
    conn = sqlite3.connect(":memory:")
    cursor = conn.cursor()
    try:
        cursor.executescript(schema_sql)
        conn.commit()
    except Exception as e:
        return None, str(e)
    return conn, None

def run_sql(conn, sql):
    try:
        cursor = conn.cursor()
        cursor.execute(sql)
        return cursor.fetchall()
    except Exception as e:
        return f"ERROR: {e}"

def generate_sql(model, tokenizer, schema, question, max_new_tokens=200):
    system_msg = "You are a helpful assistant that writes SQL queries given a database schema and a question."
    user_msg = f"### Database Schema:\n{schema}\n\n### Question:\n{question}"
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def evaluate_model(model, tokenizer, test_examples, num_samples=100):
    correct, total = 0, 0
    results = []
    for example in test_examples.select(range(min(num_samples, len(test_examples)))):
        schema, question, gold_sql = example["sql_context"], example["sql_prompt"], example["sql"]
        conn, err = create_db_from_schema(schema)
        if conn is None:
            continue
        predicted_sql = generate_sql(model, tokenizer, schema, question)
        gold_result = run_sql(conn, gold_sql)
        pred_result = run_sql(conn, predicted_sql)
        is_correct = (gold_result == pred_result) and not isinstance(pred_result, str)
        correct += int(is_correct)
        total += 1
        results.append({"question": question, "gold_sql": gold_sql, "predicted_sql": predicted_sql, "correct": is_correct})
        conn.close()
    return (correct / total if total > 0 else 0), results

## Evaluation — loading trained adapter from Hugging Face Hub

In [ ]:
from transformers import AutoTokenizer
from peft import PeftModel

adapter_id = "zain-the-npc/qwen2-7b-sql-qlora"
tokenizer = AutoTokenizer.from_pretrained(adapter_id)
base_model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.float16)
model = PeftModel.from_pretrained(base_model, adapter_id)
model.eval()